# 04 — Polished Findings

**Three findings about the Dubai property market** distilled from 1,037,522 transactions (1995–2023). Each section: hypothesis → chart → narrative → business implication.

These are the same findings featured on the [Tableau Public dashboard](https://public.tableau.com/app/profile/mohammad.mikail/viz/workbook_17794578815150/Dashboard). The notebook is the analytical source-of-record; the dashboard is the recruiter-facing artifact.

**TL;DR**:
1. Business Bay off-plan units trade at a +30–60% PREMIUM over ready stock — the opposite of conventional real-estate logic. Reason: new-launch concentration in premium Burj-area positions.
2. Dubai Marina's year-over-year price changes match every major Dubai macro event published in market reports (2009 GFC bounce, 2014 mortgage-cap peak, 2020 COVID dip, 2022 plateau) — the dataset is a credible representation of the market.
3. Meydan saw 19 + 18 new project launches in 2021–22 — Dubai's most concentrated supply pipeline — with only modest +4% 5-year price growth, hinting at absorption-side risk.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 10,
})

from src import config, db
print('db path:', config.DB_PATH)

---
## Finding 1 — Business Bay off-plan trades AT A PREMIUM to ready

**Hypothesis going in**: off-plan units should trade at a *discount* to ready stock. The buyer takes completion/handover risk (typically 2–4 years to delivery) and forgoes immediate rental income, so the market should compensate them with a lower price per sqft than equivalent ready inventory.

**What we found**: in Business Bay, the opposite is consistently true.

In [ ]:
# Pull off-plan vs ready medians for Business Bay from v_offplan_vs_ready_gap
with db.connect() as conn:
    bb = pd.read_sql("""
        SELECT g.year_quarter,
               g.median_pps_offplan, g.median_pps_ready,
               g.offplan_txn_count, g.ready_txn_count
        FROM v_offplan_vs_ready_gap g
        JOIN areas a ON a.area_id = g.area_id
        WHERE a.area_name = 'Business Bay'
          AND g.offplan_txn_count >= 30 AND g.ready_txn_count >= 30
          AND g.median_pps_offplan IS NOT NULL AND g.median_pps_ready IS NOT NULL
        ORDER BY g.year_quarter
    """, conn)

bb['gap_pct'] = 100 * (bb['median_pps_offplan'] - bb['median_pps_ready']) / bb['median_pps_ready']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Top panel: two price series
ax1.plot(bb['year_quarter'], bb['median_pps_offplan'], marker='o', markersize=5,
         label='Off-plan median price/sqft', color='#c0392b', linewidth=2)
ax1.plot(bb['year_quarter'], bb['median_pps_ready'],   marker='o', markersize=5,
         label='Ready median price/sqft',   color='#2c3e50', linewidth=2)
ax1.set_ylabel('AED / sqft')
ax1.set_title('Business Bay: off-plan vs ready median price/sqft')
ax1.legend(loc='upper left', framealpha=0.95)
ax1.grid(alpha=0.3)

# Bottom panel: gap %
colors = ['#c0392b' if g > 0 else '#27ae60' for g in bb['gap_pct']]
ax2.bar(bb['year_quarter'], bb['gap_pct'], color=colors, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.7)
ax2.set_ylabel('Off-plan vs ready (%)')
ax2.set_xlabel('Quarter')
ax2.set_title('Off-plan premium (red) / discount (green) over ready, by quarter')
ax2.tick_params(axis='x', rotation=60)
ax2.grid(alpha=0.3, axis='y')

# Annotate the persistent-premium observation
median_gap = bb['gap_pct'].median()
qtrs_premium = (bb['gap_pct'] > 0).sum()
ax2.annotate(f'Median gap: +{median_gap:.0f}%\n{qtrs_premium}/{len(bb)} quarters\nshow off-plan PREMIUM',
             xy=(0.99, 0.95), xycoords='axes fraction', ha='right', va='top',
             fontsize=9, bbox=dict(boxstyle='round', facecolor='#fff3cd',
                                    edgecolor='#856404', alpha=0.9))

plt.tight_layout()
plt.savefig('../dashboard/screenshots/finding_1_business_bay.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Quarters analyzed: {len(bb)} (each requires \u226530 off-plan + \u226530 ready transactions)')
print(f'Median off-plan premium: +{median_gap:.1f}%')
print(f'Quarters where off-plan traded above ready: {qtrs_premium}/{len(bb)}')

### Narrative

Across every quarter with sufficient transaction volume (≥30 of each type), Business Bay off-plan units trade above the area's ready inventory — by a median of ~40%. This isn't a one-quarter anomaly that could be explained by a single mega-deal; it's a structural, persistent feature of the local market.

### Why this happens (a hypothesis worth testing further)

Business Bay's recent off-plan launches have concentrated in **premium Burj-area positions** (DAMAC, Sobha, Omniyat, Ellington — all releasing high-spec products targeting capital-flight buyers from Russia/CIS and high-net-worth UAE residents). The ready inventory in Business Bay, by contrast, includes a long tail of older buildings completed in the 2010s at mid-tier specs. The per-sqft comparison isn't comparing the same product — it's comparing 2024-launch ultra-luxury vs 2014-completion mid-market.

### Business implication

**For a buyer-side analyst memo at a Dubai consultancy**: flag the off-plan premium in Business Bay as a *positioning artifact, not a fundamentals signal*. A client comparing off-plan vs ready price-per-sqft will mistakenly conclude that off-plan is overpriced; the correct comparison is by product class (specification tier and tower positioning), not by completion status. Without this caveat, off-plan investments in Business Bay look pricier than they truly are.

---
## Finding 2 — Dubai Marina's price history matches every published Dubai market cycle

**Hypothesis going in**: any analyst who reads Dubai market commentary regularly knows the major price cycle pivots — 2008 GFC peak, 2009 crash, 2014 pre-mortgage-cap peak, 2020 COVID dip, 2022 plateau. If our dataset is credible, **Dubai Marina** (the city's most-traded area) should reproduce this pattern faithfully.

**What we found**: the year-over-year median price/sqft change for Marina matches the published market history precisely. This is the data-credibility win.

In [ ]:
with db.connect() as conn:
    marina = pd.read_sql("""
        SELECT year_month, median_price_per_sqft, txn_count
        FROM v_monthly_area_medians m
        JOIN areas a ON a.area_id = m.area_id
        WHERE a.area_name = 'Marsa Dubai'
          AND m.property_type = 'Unit'
          AND m.txn_count >= 30
        ORDER BY year_month
    """, conn)

marina['date'] = pd.to_datetime(marina['year_month'])
marina = marina[marina['date'] >= '2008-01-01']  # focus on the recognizable era

# Compute trailing-12-month YoY at each month
marina['ma12'] = marina['median_price_per_sqft'].rolling(12, min_periods=6).mean()

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.plot(marina['date'], marina['median_price_per_sqft'], color='#1f77b4', alpha=0.4,
        linewidth=1, label='Monthly median')
ax.plot(marina['date'], marina['ma12'], color='#1f77b4', linewidth=2.5,
        label='12-month moving average')

events = [
    ('2008-09-15', 17000, 'Lehman collapses\n(GFC trigger)',          (-90, 30), '#c0392b'),
    ('2009-06-01', 8000,  'Marina median\nbottoms out',                ( 10, 20), '#16a085'),
    ('2013-12-28', 23000, 'CB-UAE mortgage cap\nintroduced',           (-90, 40), '#8e44ad'),
    ('2020-04-01', 13500, 'COVID dip',                                 ( 30, -25), '#34495e'),
    ('2022-09-01', 22500, '2022 plateau\n(post-Golden Visa boom)',     ( 10, 30), '#d35400'),
]
for date, y, label, offset, color in events:
    d = pd.to_datetime(date)
    ax.axvline(d, color=color, linewidth=0.7, alpha=0.4, linestyle='--')
    ax.annotate(label, xy=(d, y), xytext=offset, textcoords='offset points',
                fontsize=8.5, color=color,
                arrowprops=dict(arrowstyle='->', color=color, alpha=0.6, lw=0.8),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=color, alpha=0.85))

ax.set_title('Dubai Marina median price/sqft, 2008–2023 — every major macro event visible')
ax.set_xlabel('')
ax.set_ylabel('AED / sqft')
ax.legend(loc='lower right', framealpha=0.95)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../dashboard/screenshots/finding_2_marina_history.png', dpi=130, bbox_inches='tight')
plt.show()

# Print the key YoY numbers as a defensible table
with db.connect() as conn:
    yoy = pd.read_sql("""
        SELECT y.year_month,
               printf('%.0f', y.current_median_pps)    AS pps_current,
               printf('%.0f', y.prior_year_median_pps) AS pps_prior,
               printf('%+.1f%%', y.yoy_pct_change)     AS yoy
        FROM v_yoy_change_by_area_proptype y
        JOIN areas a ON a.area_id = y.area_id
        WHERE a.area_name = 'Marsa Dubai' AND y.property_type = 'Unit'
          AND y.year_month IN ('2009-12','2014-12','2020-12','2022-12')
        ORDER BY y.year_month
    """, conn)
print('\nKey YoY pivot points (Marina Units):')
print(yoy.to_string(index=False))

### Narrative

The 12-month moving average traces the canonical Dubai cycle: peak just before Sep 2008 (~17k AED/sqft), crash to ~8k by mid-2009, slow recovery 2010–2013, a 2014 peak around 23k that coincides with the Central Bank of UAE introducing mortgage caps (Dec 2013, applied 2014), a long 2015–2019 plateau, a brief 2020 COVID dip, and a sharp 2021–2023 rise to all-time highs ~29k AED/sqft.

The YoY pivot table reads like the table-of-contents of every published Dubai market report: +42% in 2009 (post-GFC rebound), +17% in 2014 (pre-cap peak), −20% in 2020 (COVID), +1.4% in 2022 (plateau after sharp 2021 climb).

### Business implication

**For a recruiter or analyst evaluating the dataset's credibility**: every reference point this chart shows can be cross-checked against publicly available Dubai market commentary from Knight Frank, JLL, Reidin, Property Monitor, or Bayut's research desk. The faithful reproduction validates that the ingestion + IQR cleaning + display-name layer collectively didn't distort the underlying market signal. **Any subsequent finding built on this dataset inherits the same credibility.**

---
## Finding 3 — Meydan absorbed Dubai's most concentrated supply pipeline

**Hypothesis going in**: supply pressure should be visible in either price deceleration or rent declines. We don't have rent data, but we can spot supply concentration via a *new-project-launches-per-year-per-area* proxy.

**What we found**: Meydan (DLD canonical: Al Merkadh) saw 19 new project launches in 2021 followed by 18 in 2022 — substantially ahead of any other Dubai area in that window — alongside only modest +4% 5-year price CAGR. The combination flags absorption-side risk for 2024–25.

In [ ]:
with db.connect() as conn:
    pipe = pd.read_sql("""
        SELECT COALESCE(a.display_name, a.area_name) AS area,
               s.handover_year_proxy AS year,
               s.new_projects
        FROM v_supply_pipeline_proxy s
        JOIN areas a ON a.area_id = s.area_id
        WHERE s.handover_year_proxy BETWEEN 2015 AND 2022
    """, conn)

# Top 8 areas by total launches over the window
top_areas = (pipe.groupby('area')['new_projects'].sum()
             .sort_values(ascending=False).head(8).index.tolist())
pipe_top = pipe[pipe['area'].isin(top_areas)]
pivot = pipe_top.pivot(index='area', columns='year', values='new_projects').fillna(0)
pivot = pivot.loc[top_areas]  # preserve top-down ordering

fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.0f', cbar_kws={'label': 'New projects'}, ax=ax,
            linewidths=0.5, linecolor='white', annot_kws={'size': 9, 'weight': 'semibold'})
ax.set_title('Top 8 Dubai areas by new project launches (proxy: first-transaction-year per project), 2015–2022')
ax.set_xlabel('')
ax.set_ylabel('')

# Highlight Meydan row
if 'Meydan' in pivot.index:
    meydan_row = list(pivot.index).index('Meydan')
    ax.add_patch(plt.Rectangle((0, meydan_row), len(pivot.columns), 1,
                                fill=False, edgecolor='#c0392b', lw=2.5))
    ax.annotate('Meydan: 19 + 18 launches\nin a single 2-year window',
                xy=(len(pivot.columns) + 0.6, meydan_row + 0.5),
                xytext=(len(pivot.columns) + 1.5, meydan_row - 0.5),
                fontsize=9, color='#c0392b',
                arrowprops=dict(arrowstyle='->', color='#c0392b'),
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                          edgecolor='#c0392b', alpha=0.95))

plt.tight_layout()
plt.savefig('../dashboard/screenshots/finding_3_meydan_supply.png', dpi=130, bbox_inches='tight')
plt.show()

# Pair the supply story with Meydan's price CAGR
cagr_df = pd.read_csv('../dashboard/area_5y_cagr_vs_offplan.csv')
meydan_cagr = cagr_df[cagr_df['area_name'] == 'Al Merkadh']
if len(meydan_cagr):
    print(f"\nMeydan 5-year price CAGR: +{meydan_cagr.iloc[0]['cagr_5y_pct']:.2f}%")
    print(f"Off-plan premium vs ready (recent): +{meydan_cagr.iloc[0]['current_offplan_gap_pct']:.1f}%")
    print(f"Recent-window transactions: {int(meydan_cagr.iloc[0]['recent_txn_count'])}")

### Narrative

Meydan led the 2021–22 launch wave with 37 new projects across two years, more than any other top-10 Dubai area in the same window. Yet its 5-year price CAGR (~+4%) is modest compared to Marina (+9.3%) or Palm Jumeirah (+9.6%). The pattern is straightforward in classical real estate terms: supply growth is outpacing demand absorption.

Two further data points reinforce this. First, Meydan's off-plan-vs-ready premium is +25% — well below Business Bay (+40%) or Marina (+80%) — suggesting the local market hasn't yet priced new launches at the same premium tier as more established areas. Second, the new-launches-per-year shape (15 → 13 → ~12 → 19 → 18) signals a developer pipeline that ramped up well before any visible demand absorption test.

### Business implication

**For a buyer-side or fund-side memo**: Meydan is the area where over-supply risk is most easily quantifiable. If a client asks "where in Dubai should I delay a 2024 entry until I see absorption proof?" — Meydan is the area with the strongest *quantitative* answer. The combination of (a) high recent launch volume, (b) modest price acceleration, and (c) a compressed off-plan premium is the textbook signature of a market that may need a year or two to absorb existing supply before resuming sustainable price growth.

---
## Limitations

- **Dataset ends Feb 2023.** All findings reflect the market state up to that point; 2023–2025 dynamics are not in scope. Updating with DLD 2023+ downloads is a stretch goal documented in the README.
- **No rent data in source.** The originally-planned "rental yield" finding was dropped. The off-plan-vs-ready gap (Finding 1) is the substitute headline finding and is arguably stronger as a market-microstructure signal.
- **Supply pipeline is a proxy.** `v_supply_pipeline_proxy` infers handover year from first transaction year per project. This is a reasonable proxy but conflates off-plan launch with handover for projects that sold-out at launch. Real handover data would tighten Finding 3.
- **Median-of-medians weighting in dashboard ≠ true grand median.** The Tableau dashboard uses a transaction-weighted average of per-area medians for Dubai-wide trend lines. This notebook uses per-area medians directly, which is the more defensible analytical primitive.

## What's next (Week 4)

- 90-second Loom walkthrough of the dashboard
- Medium article (~1000 words) expanding Finding 1 (Business Bay) into a full "why this matters for Dubai investors" essay
- LinkedIn post highlighting the off-plan-share-as-sentiment-gauge angle (the deferred 4th finding)